# 02 - Pipeline ETL (CSV → SQLite → tabla curada)

Flujo:
1. **Extracción:** leer el CSV original y cargarlo tal cual a la tabla `prestamos_raw` en SQLite.
2. **Transformación:** consultar `prestamos_raw` **desde la base de datos** con Python y limpiar/transformar.
3. **Carga:** guardar el resultado en la tabla `prestamos_curados` (+ vistas `vw_entrenamiento` y `vw_prueba`).

El pipeline es reproducible: cada vez que se corre borra y recrea las tablas de datos (usa `database/schema.sql`).
La bitácora `etl_log` no se borra: va guardando una fila por cada ejecución.

In [1]:
import sqlite3
from datetime import datetime

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

RUTA_CSV = "../data/loan_sanction_train.csv"
RUTA_BD = "../database/prestamos.db"
RUTA_SCHEMA = "../database/schema.sql"
SEMILLA = 42

## 1. Extracción
### 1.1 Lectura del dataset original

In [2]:
df_csv = pd.read_csv(RUTA_CSV)
print("Filas leídas del CSV:", len(df_csv))
df_csv.head(3)

Filas leídas del CSV: 614


,Loan_ID,Gender,Married,Dependents,Education,Self_Employed,ApplicantIncome,CoapplicantIncome,LoanAmount,Loan_Amount_Term,Credit_History,Property_Area,Loan_Status
0,LP001002,Male,No,0,Graduate,No,5849,0.0,NaN,360.0,1.0,Urban,Y
1,LP001003,Male,Yes,1,Graduate,No,4583,1508.0,128.0,360.0,1.0,Rural,N
2,LP001005,Male,Yes,0,Graduate,Yes,3000,0.0,66.0,360.0,1.0,Urban,Y


### 1.2 Conexión a la base de datos y creación de tablas

In [3]:
conexion = sqlite3.connect(RUTA_BD)
conexion.execute("PRAGMA foreign_keys = ON")

with open(RUTA_SCHEMA, encoding="utf-8") as f:
    conexion.executescript(f.read())

# tablas creadas
pd.read_sql("SELECT type, name FROM sqlite_master WHERE type IN ('table','view') ORDER BY type, name", conexion)

,type,name
0,table,etl_log
1,table,metadatos_columnas
2,table,prestamos_curados
3,table,prestamos_raw
4,table,sqlite_sequence
5,view,vw_entrenamiento
6,view,vw_prueba
7,view,vw_resumen_aprobacion


### 1.3 Carga inicial (datos originales) a `prestamos_raw`
Se renombran las columnas a snake_case y los nulos de pandas (`NaN`) pasan a `NULL` en SQL.

In [4]:
nombres_bd = {
    "Loan_ID": "loan_id", "Gender": "gender", "Married": "married", "Dependents": "dependents",
    "Education": "education", "Self_Employed": "self_employed", "ApplicantIncome": "applicant_income",
    "CoapplicantIncome": "coapplicant_income", "LoanAmount": "loan_amount",
    "Loan_Amount_Term": "loan_amount_term", "Credit_History": "credit_history",
    "Property_Area": "property_area", "Loan_Status": "loan_status",
}
df_raw = df_csv.rename(columns=nombres_bd)
df_raw = df_raw.astype(object).where(df_raw.notna(), None)  # NaN -> None (NULL)

columnas = list(df_raw.columns)
sql_insert = f"INSERT INTO prestamos_raw ({', '.join(columnas)}) VALUES ({', '.join(['?'] * len(columnas))})"
conexion.executemany(sql_insert, df_raw.values.tolist())
conexion.commit()

pd.read_sql("SELECT COUNT(*) AS filas_en_raw FROM prestamos_raw", conexion)

,filas_en_raw
0,614


### 1.4 Consulta desde la base de datos usando Python
A partir de aquí **ya no se usa el CSV**: todo sale de la base de datos.

In [5]:
datos = pd.read_sql("SELECT * FROM prestamos_raw", conexion)
print(datos.shape)
datos.isna().sum()

(614, 13)


loan_id                0
gender                13
married                3
dependents            15
education              0
self_employed         32
applicant_income       0
coapplicant_income     0
loan_amount           22
loan_amount_term      14
credit_history        50
property_area          0
loan_status            0
dtype: int64

In [6]:
# validación rápida: lo que hay en la BD es lo mismo que en el CSV
assert len(datos) == len(df_csv)
assert datos["loan_status"].value_counts().equals(df_csv["Loan_Status"].value_counts().rename("count").rename_axis("loan_status"))
print("OK: la tabla raw coincide con el CSV")

OK: la tabla raw coincide con el CSV


## 2. Transformación

### 2.1 Separar train / test antes de imputar (evitar data leakage)
Los valores para rellenar nulos (moda, mediana) se calculan **solo con train**. Si se calcularan
con todo el dataset, información del set de prueba se "filtraría" al entrenamiento.
La separación es estratificada (mantiene la proporción 69/31) y queda guardada en la columna `conjunto`.

In [7]:
ids_train, ids_test = train_test_split(
    datos["loan_id"], test_size=0.2, stratify=datos["loan_status"], random_state=SEMILLA
)
datos["conjunto"] = np.where(datos["loan_id"].isin(ids_train), "train", "test")
pd.crosstab(datos["conjunto"], datos["loan_status"], margins=True)

loan_status,N,Y,All
conjunto,,,
test,38,85,123
train,154,337,491
All,192,422,614


### 2.2 Conversión de tipos

In [8]:
datos["dependents"] = datos["dependents"].replace("3+", "3")
datos["dependents"] = pd.to_numeric(datos["dependents"])  # sigue con NaN, se imputa abajo
datos[["dependents", "credit_history", "loan_amount_term"]].dtypes

dependents          float64
credit_history      float64
loan_amount_term    float64
dtype: object

### 2.3 Tratamiento de valores nulos

| Columna | Estrategia | Por qué |
|---|---|---|
| gender, married, self_employed, dependents | Moda de train | Pocos nulos, variables categóricas |
| loan_amount | Mediana de train | Variable con outliers: la mediana es más robusta que la media |
| loan_amount_term | Moda de train (360) | Casi todos los préstamos son a 360 meses |
| credit_history | **No se imputa**: se crea la bandera `historial_desconocido` | En el EDA los nulos tienen una tasa de aprobación propia (74%) |

In [9]:
train = datos[datos["conjunto"] == "train"]

valores_relleno = {
    "gender": train["gender"].mode()[0],
    "married": train["married"].mode()[0],
    "self_employed": train["self_employed"].mode()[0],
    "dependents": train["dependents"].mode()[0],
    "loan_amount": train["loan_amount"].median(),
    "loan_amount_term": train["loan_amount_term"].mode()[0],
}
print("Valores de relleno (calculados solo con train):")
valores_relleno

Valores de relleno (calculados solo con train):


{'gender': 'Male',
 'married': 'Yes',
 'self_employed': 'No',
 'dependents': np.float64(0.0),
 'loan_amount': np.float64(128.0),
 'loan_amount_term': np.float64(360.0)}

In [10]:
datos = datos.fillna(valores_relleno)

datos["historial_desconocido"] = datos["credit_history"].isna().astype(int)
datos["historial_bueno"] = (datos["credit_history"] == 1).astype(int)
# historial malo = las dos banderas en 0

datos.drop(columns="credit_history").isna().sum().sum()

np.int64(0)

### 2.4 Codificación de variables categóricas
- Variables de 2 valores → 0/1.
- `property_area` tiene 3 valores → one-hot quitando una categoría (Rural queda como base).

In [11]:
datos["es_hombre"] = (datos["gender"] == "Male").astype(int)
datos["casado"] = (datos["married"] == "Yes").astype(int)
datos["graduado"] = (datos["education"] == "Graduate").astype(int)
datos["independiente"] = (datos["self_employed"] == "Yes").astype(int)
datos["zona_semiurbana"] = (datos["property_area"] == "Semiurban").astype(int)
datos["zona_urbana"] = (datos["property_area"] == "Urban").astype(int)
datos["dependientes"] = datos["dependents"].astype(int)
datos["aprobado"] = (datos["loan_status"] == "Y").astype(int)

### 2.5 Variables derivadas y transformación logarítmica
- Ingresos y monto tienen colas largas → `log(1 + x)`.
- `cuota_aprox`: monto / plazo (cuánto paga por mes, en miles).
- `relacion_cuota_ingreso`: qué tan pesada es la cuota comparada con el ingreso total del hogar.

In [12]:
ingreso_total = datos["applicant_income"] + datos["coapplicant_income"]

datos["plazo_meses"] = datos["loan_amount_term"]
datos["log_ingreso_solicitante"] = np.log1p(datos["applicant_income"])
datos["log_ingreso_coaplicante"] = np.log1p(datos["coapplicant_income"])
datos["log_ingreso_total"] = np.log1p(ingreso_total)
datos["log_monto_prestamo"] = np.log1p(datos["loan_amount"])
datos["cuota_aprox"] = datos["loan_amount"] / datos["plazo_meses"]
datos["relacion_cuota_ingreso"] = datos["cuota_aprox"] * 1000 / ingreso_total

datos[["log_ingreso_total", "log_monto_prestamo", "cuota_aprox", "relacion_cuota_ingreso"]].describe().round(3)

,log_ingreso_total,log_monto_prestamo,cuota_aprox,relacion_cuota_ingreso
count,614.000,614.000,614.000,614.000
mean,8.670,4.866,0.474,0.077
std,0.545,0.491,0.511,0.079
min,7.274,2.303,0.025,0.007
25%,8.335,4.618,0.289,0.055
50%,8.597,4.860,0.361,0.069
75%,8.926,5.110,0.506,0.082
max,11.302,6.553,9.250,1.237


### 2.6 Escalamiento
**No** se escala aquí. El escalado (`StandardScaler`) se aplica dentro del pipeline del modelo
(notebook 03), ajustado solo con train. Así la tabla curada queda en unidades entendibles
y se evita leakage. Los modelos de árboles no lo necesitan; la regresión logística y KNN sí.

### 2.7 Eliminación de columnas irrelevantes / originales
- Se eliminan las columnas de texto originales (ya están codificadas).
- Se eliminan los ingresos y montos en escala original (quedan sus versiones log).
- `loan_id` se mantiene **solo como llave** de la tabla, no como predictor.

In [13]:
predictoras = [
    "es_hombre", "casado", "dependientes", "graduado", "independiente",
    "zona_semiurbana", "zona_urbana", "historial_bueno", "historial_desconocido",
    "plazo_meses", "log_ingreso_solicitante", "log_ingreso_coaplicante",
    "log_ingreso_total", "log_monto_prestamo", "cuota_aprox", "relacion_cuota_ingreso",
]
objetivo = "aprobado"

curados = datos[["loan_id", "conjunto"] + predictoras + [objetivo]].copy()
print(curados.shape)
curados.head()

(614, 19)


,loan_id,conjunto,es_hombre,casado,dependientes,graduado,independiente,zona_semiurbana,zona_urbana,historial_bueno,historial_desconocido,plazo_meses,log_ingreso_solicitante,log_ingreso_coaplicante,log_ingreso_total,log_monto_prestamo,cuota_aprox,relacion_cuota_ingreso,aprobado
0,LP001002,train,1,0,0,1,0,0,1,1,0,360.0,8.674197,0.000000,8.674197,4.859812,0.355556,0.060789,1
1,LP001003,train,1,1,1,1,0,0,0,1,0,360.0,8.430327,7.319202,8.714732,4.859812,0.355556,0.058374,0
2,LP001005,test,1,1,0,1,1,0,1,1,0,360.0,8.006701,0.000000,8.006701,4.204693,0.183333,0.061111,1
3,LP001006,train,1,1,0,0,0,0,1,1,0,360.0,7.857094,7.765993,8.505525,4.795791,0.333333,0.067463,1
4,LP001008,train,1,0,0,1,0,0,1,1,0,360.0,8.699681,0.000000,8.699681,4.955827,0.391667,0.065278,1


### 2.8 Separación de variables predictoras y variable objetivo

In [14]:
X = curados[predictoras]
y = curados[objetivo]

# controles contra data leakage
assert objetivo not in X.columns
assert "loan_status" not in X.columns and "loan_id" not in X.columns
assert curados.isna().sum().sum() == 0
print("X:", X.shape, "| y:", y.shape, "| sin nulos, objetivo fuera de X")

X: (614, 16) | y: (614,) | sin nulos, objetivo fuera de X


## 3. Carga a la capa final
### 3.1 Tabla curada `prestamos_curados`

In [15]:
curados.to_sql("prestamos_curados", conexion, if_exists="append", index=False)

614

### 3.2 Diccionario de columnas
El notebook de entrenamiento lee de aquí cuáles columnas son predictoras, así no se escriben a mano dos veces.

In [16]:
descripciones = {
    "loan_id": ("llave", "Identificador de la solicitud"),
    "conjunto": ("control", "train o test (split estratificado, semilla 42)"),
    "es_hombre": ("predictora", "1 si Gender = Male"),
    "casado": ("predictora", "1 si Married = Yes"),
    "dependientes": ("predictora", "Número de dependientes (3+ = 3)"),
    "graduado": ("predictora", "1 si Education = Graduate"),
    "independiente": ("predictora", "1 si Self_Employed = Yes"),
    "zona_semiurbana": ("predictora", "1 si Property_Area = Semiurban (base: Rural)"),
    "zona_urbana": ("predictora", "1 si Property_Area = Urban (base: Rural)"),
    "historial_bueno": ("predictora", "1 si Credit_History = 1"),
    "historial_desconocido": ("predictora", "1 si Credit_History es nulo"),
    "plazo_meses": ("predictora", "Plazo del préstamo en meses"),
    "log_ingreso_solicitante": ("predictora", "log(1 + ApplicantIncome)"),
    "log_ingreso_coaplicante": ("predictora", "log(1 + CoapplicantIncome)"),
    "log_ingreso_total": ("predictora", "log(1 + ingreso total del hogar)"),
    "log_monto_prestamo": ("predictora", "log(1 + LoanAmount)"),
    "cuota_aprox": ("predictora", "LoanAmount / plazo (miles por mes)"),
    "relacion_cuota_ingreso": ("predictora", "cuota_aprox * 1000 / ingreso total"),
    "aprobado": ("objetivo", "1 si Loan_Status = Y"),
}
conexion.executemany(
    "INSERT INTO metadatos_columnas (columna, rol, descripcion) VALUES (?, ?, ?)",
    [(c, r, d) for c, (r, d) in descripciones.items()],
)

### 3.3 Bitácora de la ejecución

In [17]:
conexion.execute(
    """INSERT INTO etl_log (fecha, filas_leidas, filas_raw, filas_curadas, filas_train, filas_test, nulos_restantes)
       VALUES (?, ?, ?, ?, ?, ?, ?)""",
    (datetime.now().isoformat(timespec="seconds"), len(df_csv),
     conexion.execute("SELECT COUNT(*) FROM prestamos_raw").fetchone()[0],
     len(curados), int((curados.conjunto == "train").sum()), int((curados.conjunto == "test").sum()),
     int(curados.isna().sum().sum())),
)
conexion.commit()

## 4. Verificación de la carga (consultas SQL)

In [18]:
pd.read_sql("SELECT * FROM etl_log", conexion)

,id_ejecucion,fecha,filas_leidas,filas_raw,filas_curadas,filas_train,filas_test,nulos_restantes
0,1,2026-11-23T15:53:28,614,614,614,491,123,0


In [19]:
pd.read_sql("""
SELECT conjunto, COUNT(*) AS filas, ROUND(AVG(aprobado) * 100, 1) AS pct_aprobado
FROM prestamos_curados GROUP BY conjunto
""", conexion)

,conjunto,filas,pct_aprobado
0,test,123,69.1
1,train,491,68.6


In [20]:
pd.read_sql("SELECT * FROM vw_resumen_aprobacion ORDER BY property_area, historial", conexion)

,property_area,historial,solicitudes,pct_aprobado
0,Rural,cumple,137,71.5
1,Rural,no cumple,28,7.1
2,Rural,sin historial,14,71.4
3,Semiurban,cumple,187,87.7
4,Semiurban,no cumple,30,13.3
5,Semiurban,sin historial,16,68.8
6,Urban,cumple,151,76.8
7,Urban,no cumple,31,3.2
8,Urban,sin historial,20,80.0


In [21]:
# esquema final de la tabla curada
pd.read_sql("PRAGMA table_info(prestamos_curados)", conexion)[["name", "type", "notnull", "pk"]]

,name,type,notnull,pk
0,loan_id,TEXT,0,1
1,conjunto,TEXT,1,0
2,es_hombre,INTEGER,1,0
3,casado,INTEGER,1,0
4,dependientes,INTEGER,1,0
5,graduado,INTEGER,1,0
6,independiente,INTEGER,1,0
7,zona_semiurbana,INTEGER,1,0
8,zona_urbana,INTEGER,1,0
9,historial_bueno,INTEGER,1,0


In [22]:
conexion.close()
print("ETL terminado. Base de datos:", RUTA_BD)

ETL terminado. Base de datos: ../database/prestamos.db
